In [ ]:
evidencia = {
  "caso": {
    "tienda_id": "T0001",
    "producto_id": "P001",
    "fecha_corte": "2026-10-01",
    "dias_consecutivos_sin_inventario": 3,
    "unidades_vendidas_28d_previos": 145,
    "inventario_actual": 0,
    "precio_lista": 25.26,
    "cumple_criterios_m2": True
  },
  "fuentes": [
    {
      "archivo": "inventarios.csv",
      "fecha_corte": "2026-10-01"
    },
    {
      "archivo": "ventas.csv",
      "ventana_dias": 28
    }
  ]
}

In [ ]:

import json
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")

def estimar_tokens(contenido):
    if not isinstance(contenido, str):
        contenido = json.dumps(
            contenido,
            ensure_ascii=False,
            separators=(",", ":")
        )
    return len(encoding.encode(contenido))

# Sustituir por un caso real extraído de los CSV
tokens_evidencia = estimar_tokens(evidencia)

print("Tokens estimados:", tokens_evidencia)


Tokens estimados: 111


In [ ]:
tokens_evidencia_5_casos = estimar_tokens([evidencia] * 5)
print("Tokens estimados:", tokens_evidencia_5_casos)

Tokens estimados: 553


In [ ]:
import io, json, zipfile
from pathlib import Path
import pandas as pd
import tiktoken

UMBRAL_DEMANDA = 103
DIAS_MINIMOS = 2
DIAS_DEMANDA = 28
DIAS_BUSQUEDA = 14    # solo búsqueda de quiebres recientes, no demanda
PRESUPUESTO_EVIDENCIA = 660
ENCODING = 'o200k_base'  # aproximación: no es un conteo real de Luna

try:
    from google.colab import files
    print('Sube Construye_3.zip')
    uploaded = files.upload()
    zip_bytes = next((contenido for nombre, contenido in uploaded.items() if nombre.lower().endswith('.zip')), None)
    if zip_bytes is None:
        raise ValueError('No se subió un archivo ZIP.')
    z = zipfile.ZipFile(io.BytesIO(zip_bytes))
except ImportError:
    z = zipfile.ZipFile('Construye_3.zip')  # ejecución local


def cargar_csv(nombre):
    ruta = next((n for n in z.namelist() if n.endswith('/datos/' + nombre + '.csv')), None)
    if ruta is None:
        raise FileNotFoundError(f'No encontré {nombre}.csv en el ZIP')
    df = pd.read_csv(z.open(ruta))
    if 'fecha' in df.columns:
        df['fecha'] = pd.to_datetime(df['fecha'])
    return df

inventarios = cargar_csv('inventarios')
ventas = cargar_csv('ventas')
productos = cargar_csv('productos')
corte = inventarios['fecha'].max()
print(f'Corte del dataset: {corte.date()}')

# Un día por tienda-producto; si aparecen fuentes múltiples se usa el
# mínimo inventario disponible y se señala ese supuesto de agregación.
inv_dia = (inventarios.groupby(['tienda_id','producto_id','fecha'], as_index=False)
           .agg(unidades_disponibles=('unidades_disponibles','min')))
reciente = inv_dia.loc[inv_dia.fecha >= corte - pd.Timedelta(days=DIAS_BUSQUEDA-1)].copy()
reciente = reciente.sort_values(['tienda_id','producto_id','fecha'])
reciente['sin_inventario'] = reciente.unidades_disponibles.le(0)
# Nueva racha cuando cambia la pareja, falta un día, o se recupera inventario.
misma_pareja = reciente[['tienda_id','producto_id']].eq(reciente[['tienda_id','producto_id']].shift()).all(axis=1)
dia_siguiente = reciente.fecha.diff().eq(pd.Timedelta(days=1))
continua = misma_pareja & dia_siguiente & reciente.sin_inventario & reciente.sin_inventario.shift(fill_value=False)
reciente['racha_id'] = (~continua).cumsum()
rachas = (reciente.loc[reciente.sin_inventario]
          .groupby(['tienda_id','producto_id','racha_id'], as_index=False)
          .agg(inicio=('fecha','min'), fin=('fecha','max'), dias=('fecha','size')))
rachas = rachas.loc[rachas.dias >= DIAS_MINIMOS].copy()
if rachas.empty:
    raise RuntimeError('No hay rachas elegibles; revisa los datos o DIAS_BUSQUEDA.')

# Sumar demanda ANTES del inicio de la racha, no desde la fecha de corte.
# Se evalúa cada candidata en orden determinista, sin prioridad comercial.
rachas = rachas.sort_values(['tienda_id','producto_id','inicio','fin'])
caso = None
v_caso = None
for r in rachas.itertuples(index=False):
    v = ventas.loc[
        (ventas.tienda_id == r.tienda_id) &
        (ventas.producto_id == r.producto_id) &
        (ventas.fecha >= r.inicio - pd.Timedelta(days=DIAS_DEMANDA)) &
        (ventas.fecha < r.inicio)
    ].copy()
    demanda = int(v.unidades.sum())
    if demanda >= UMBRAL_DEMANDA:
        caso, v_caso = r, v
        break
if caso is None:
    raise RuntimeError('Existen rachas de inventario, pero ninguna cumple la demanda previa >= 103.')

i_caso = inventarios.loc[
    (inventarios.tienda_id == caso.tienda_id) &
    (inventarios.producto_id == caso.producto_id) &
    (inventarios.fecha >= caso.inicio) &
    (inventarios.fecha <= caso.fin)
].copy()

# Todos los registros pertinentes al MISMO caso; no se transmite el dataset completo.
def registros(df, columnas):
    lista = df[columnas].copy()
    if 'fecha' in lista:
        lista['fecha'] = lista.fecha.dt.strftime('%Y-%m-%d')
    return lista.to_dict(orient='records')

original = {
    'tienda_id': str(caso.tienda_id),
    'producto_id': str(caso.producto_id),
    'inventarios': registros(i_caso, ['fecha','unidades_disponibles','fuente']),
    'ventas': registros(v_caso, ['fecha','unidades','precio_efectivo']),
}
producto = productos.loc[productos.producto_id.eq(caso.producto_id)]
precio_lista = float(producto.iloc[0]['precio_lista']) if not producto.empty else None

resumido = {
    'tienda_id': str(caso.tienda_id),
    'producto_id': str(caso.producto_id),
    'inicio_quiebre': caso.inicio.strftime('%Y-%m-%d'),
    'fin_racha_observada': caso.fin.strftime('%Y-%m-%d'),
    'dias_consecutivos_sin_inventario': int(caso.dias),
    'unidades_vendidas_28d_previos': int(v_caso.unidades.sum()),
    'ventana_ventas': {
        'desde': (caso.inicio - pd.Timedelta(days=DIAS_DEMANDA)).strftime('%Y-%m-%d'),
        'hasta_exclusivo': caso.inicio.strftime('%Y-%m-%d'),
    },
    'precio_lista': precio_lista,
    'fuentes': [
        {'archivo': 'inventarios.csv', 'fecha_corte': corte.strftime('%Y-%m-%d')},
        {'archivo': 'ventas.csv', 'ventana_dias': DIAS_DEMANDA},
        {'archivo': 'productos.csv'},
    ],
    'cumple_criterios_m2': True,
}

enc = tiktoken.get_encoding(ENCODING)
def serializar(obj):
    return json.dumps(obj, ensure_ascii=False, separators=(',',':'))
def contar(obj):
    return len(enc.encode(serializar(obj)))

antes, despues = contar(original), contar(resumido)
print('\n--- CASO ELEGIBLE (sin ranking comercial) ---')
print(f'Tienda: {caso.tienda_id} | Producto: {caso.producto_id}')
print(f'Racha: {caso.inicio.date()} a {caso.fin.date()} ({caso.dias} días)')
print(f'Demanda previa de 28 días: {int(v_caso.unidades.sum())} unidades')
print(f'Registros de inventario incluidos: {len(i_caso)}')
print(f'Registros de ventas incluidos: {len(v_caso)}')
print('\n--- ESTIMACIÓN DE TOKENS ---')
print(f'Original: {antes:,} tokens')
print(f'Resumido: {despues:,} tokens')
print(f'Reducción: {antes - despues:,} tokens ({(antes-despues)/antes:.1%})')
print(f'¿Resumen dentro de {PRESUPUESTO_EVIDENCIA} tokens de evidencia?: {despues <= PRESUPUESTO_EVIDENCIA}')
print('\n--- JSON RESUMIDO ---')
print(json.dumps(resumido, indent=2, ensure_ascii=False))
print('\nNotas: mide solo el JSON de evidencia, no instrucciones ni respuesta.')
print('Los datos fuente se usan transitoriamente; no persistir información protegida del portal.')
print('La cifra de tokens depende del tokenizador; no representa el consumo real de Luna.')


Sube Construye_3.zip


Saving Construye_3.zip to Construye_3.zip
Corte del dataset: 2026-08-30

--- CASO ELEGIBLE (sin ranking comercial) ---
Tienda: T0001 | Producto: P028
Racha: 2026-08-29 a 2026-08-30 (2 días)
Demanda previa de 28 días: 108 unidades
Registros de inventario incluidos: 2
Registros de ventas incluidos: 26

--- ESTIMACIÓN DE TOKENS ---
Original: 692 tokens
Resumido: 149 tokens
Reducción: 543 tokens (78.5%)
¿Resumen dentro de 660 tokens de evidencia?: True

--- JSON RESUMIDO ---
{
  "tienda_id": "T0001",
  "producto_id": "P028",
  "inicio_quiebre": "2026-08-29",
  "fin_racha_observada": "2026-08-30",
  "dias_consecutivos_sin_inventario": 2,
  "unidades_vendidas_28d_previos": 108,
  "ventana_ventas": {
    "desde": "2026-08-01",
    "hasta_exclusivo": "2026-08-29"
  },
  "precio_lista": 37.35,
  "fuentes": [
    {
      "archivo": "inventarios.csv",
      "fecha_corte": "2026-08-30"
    },
    {
      "archivo": "ventas.csv",
      "ventana_dias": 28
    },
    {
      "archivo": "productos.csv